# HW13 — Токенизация текста, инференс BERT и fine-tuning для классификации

**Датасет:** `emotion` (6 классов: sadness, joy, love, anger, fear, surprise)  
**Модель:** `distilbert-base-uncased`  
**Задача:** классификация текстов по эмоции

## 1. Импорты, seed и среда

In [1]:
import random
import os
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Работаем офлайн — датасеты и модели уже в кеше
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"torch: {torch.__version__}")

/Users/v.razon/Desktop/вуз/ml/mirea-aie-project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Device: cpu
torch: 2.8.0


## 2. Данные и первичный анализ

In [2]:
dataset = load_dataset("emotion")
print(dataset)

Using the latest cached version of the dataset since emotion couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'split' at /Users/v.razon/.cache/huggingface/datasets/emotion/split/0.0.0/cab853a1dbdf4c42c2b3ef2173804746df8825fe (last modified on Sun Apr  5 11:47:47 2026).


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


In [3]:
label_names = dataset["train"].features["label"].names
num_labels = len(label_names)
print(f"Классы ({num_labels}): {label_names}")
print(f"Train:      {len(dataset['train'])}")
print(f"Validation: {len(dataset['validation'])}")
print(f"Test:       {len(dataset['test'])}")

Классы (6): ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
Train:      16000
Validation: 2000
Test:       2000


In [4]:
# Примеры текстов и меток
print("Примеры из train:")
for i in range(5):
    ex = dataset["train"][i]
    print(f"  [{label_names[ex['label']]}] {ex['text']}")
    print()

Примеры из train:
  [sadness] i didnt feel humiliated

  [sadness] i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake

  [anger] im grabbing a minute to post i feel greedy wrong

  [love] i am ever feeling nostalgic about the fireplace i will know that it is still on the property

  [anger] i am feeling grouchy



In [5]:
# Распределение классов в train
counts = Counter(dataset["train"]["label"])
print("Распределение классов в train:")
for label_id in sorted(counts):
    n = counts[label_id]
    print(f"  {label_names[label_id]}: {n} ({n/len(dataset['train'])*100:.1f}%)")

Распределение классов в train:
  sadness: 4666 (29.2%)
  joy: 5362 (33.5%)
  love: 1304 (8.2%)
  anger: 2159 (13.5%)
  fear: 1937 (12.1%)
  surprise: 572 (3.6%)


## 3. Токенизация

In [6]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)

print(f"Tokenizer: {type(tokenizer).__name__}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Special tokens: {tokenizer.special_tokens_map}")

Tokenizer: DistilBertTokenizerFast
Vocab size: 30522
Special tokens: {'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}


In [7]:
# Разбор токенизации на нескольких примерах
sample_texts = [
    dataset["train"][0]["text"],
    dataset["train"][1]["text"],
    dataset["train"][2]["text"],
    "I am so happy today!",
    "This is absolutely terrible and I feel awful.",
]

for text in sample_texts:
    encoded = tokenizer(text, padding=False, truncation=True, max_length=128)
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])
    print(f"Text: {text}")
    print(f"  Tokens: {tokens}")
    print(f"  input_ids: {encoded['input_ids']}")
    print(f"  attention_mask: {encoded['attention_mask']}")
    print()

Text: i didnt feel humiliated
  Tokens: ['[CLS]', 'i', 'didn', '##t', 'feel', 'humiliated', '[SEP]']
  input_ids: [101, 1045, 2134, 2102, 2514, 26608, 102]
  attention_mask: [1, 1, 1, 1, 1, 1, 1]

Text: i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake
  Tokens: ['[CLS]', 'i', 'can', 'go', 'from', 'feeling', 'so', 'hopeless', 'to', 'so', 'damned', 'hopeful', 'just', 'from', 'being', 'around', 'someone', 'who', 'cares', 'and', 'is', 'awake', '[SEP]']
  input_ids: [101, 1045, 2064, 2175, 2013, 3110, 2061, 20625, 2000, 2061, 9636, 17772, 2074, 2013, 2108, 2105, 2619, 2040, 14977, 1998, 2003, 8300, 102]
  attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Text: im grabbing a minute to post i feel greedy wrong
  Tokens: ['[CLS]', 'im', 'grabbing', 'a', 'minute', 'to', 'post', 'i', 'feel', 'greedy', 'wrong', '[SEP]']
  input_ids: [101, 10047, 9775, 1037, 3371, 2000, 2695, 1045, 2514, 20505, 3308, 1

In [8]:
# Пример padding и truncation
texts_pair = [
    "I feel great!",
    "This is a much longer sentence that contains many words and should demonstrate how padding and truncation work when texts of very different lengths are batched together.",
]

encoded_padded = tokenizer(texts_pair, padding=True, truncation=True, max_length=32, return_tensors="pt")

for i, text in enumerate(texts_pair):
    tokens = tokenizer.convert_ids_to_tokens(encoded_padded["input_ids"][i])
    print(f"Text: {text}")
    print(f"  Tokens: {tokens}")
    print(f"  attention_mask: {encoded_padded['attention_mask'][i].tolist()}")
    pad_count = tokens.count("[PAD]")
    truncated = len(tokenizer.encode(text, add_special_tokens=True)) > 32
    print(f"  PAD tokens: {pad_count}, Truncated: {truncated}")
    print()

Text: I feel great!
  Tokens: ['[CLS]', 'i', 'feel', 'great', '!', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
  attention_mask: [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  PAD tokens: 26, Truncated: False

Text: This is a much longer sentence that contains many words and should demonstrate how padding and truncation work when texts of very different lengths are batched together.
  Tokens: ['[CLS]', 'this', 'is', 'a', 'much', 'longer', 'sentence', 'that', 'contains', 'many', 'words', 'and', 'should', 'demonstrate', 'how', 'pad', '##ding', 'and', 'tr', '##un', '##cation', 'work', 'when', 'texts', 'of', 'very', 'different', 'lengths', 'are', 'batch', '##ed', '[SEP]']
  attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

## 4. Инференс готовой pretrained модели

In [9]:
# Готовая модель — sentiment analysis (POSITIVE/NEGATIVE)
# НЕ обучена на 6-классовую задачу эмоций
pretrained_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    tokenizer=AutoTokenizer.from_pretrained(
        "distilbert-base-uncased-finetuned-sst-2-english", local_files_only=True
    ),
    model_kwargs={"local_files_only": True},
    device=device,
)

inference_texts = [
    "I am so happy and grateful for everything!",
    "I feel really sad and lonely today.",
    "This makes me extremely angry!",
    "I am terrified of what might happen.",
    "What a wonderful surprise, I love it!",
]
expected = ["joy", "sadness", "anger", "fear", "surprise/love"]

print("Инференс готовой модели (sentiment-analysis, бинарная):")
print()
results = pretrained_pipe(inference_texts)
for text, result, exp in zip(inference_texts, results, expected):
    print(f"  Text: {text}")
    print(f"  Ожидаемая эмоция: {exp}")
    print(f"  Prediction: {result['label']} (score: {result['score']:.4f})")
    print()

Device set to use cpu


Инференс готовой модели (sentiment-analysis, бинарная):

  Text: I am so happy and grateful for everything!
  Ожидаемая эмоция: joy
  Prediction: POSITIVE (score: 0.9999)

  Text: I feel really sad and lonely today.
  Ожидаемая эмоция: sadness
  Prediction: NEGATIVE (score: 0.9981)

  Text: This makes me extremely angry!
  Ожидаемая эмоция: anger
  Prediction: NEGATIVE (score: 0.9992)

  Text: I am terrified of what might happen.
  Ожидаемая эмоция: fear
  Prediction: NEGATIVE (score: 0.9989)

  Text: What a wonderful surprise, I love it!
  Ожидаемая эмоция: surprise/love
  Prediction: POSITIVE (score: 0.9999)



Готовая модель `distilbert-base-uncased-finetuned-sst-2-english` обучена на бинарный sentiment analysis (POSITIVE/NEGATIVE) и не различает конкретные эмоции (sadness, anger, fear, joy, love, surprise). Она фиксирует только общую тональность, что недостаточно для 6-классовой задачи. Необходим fine-tuning.

## 5. Fine-tuning для классификации

In [10]:
MAX_LENGTH = 128

# Подготовка токенизации датасета для fine-tuning
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print(f"Tokenized dataset: {tokenized_dataset}")

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenized dataset: DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 2000
    })
})


In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    local_files_only=True,
)
model.to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model parameters: 66,958,086


In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_macro": f1}

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
)

In [13]:
trainer.train()

/Users/v.razon/Desktop/вуз/ml/mirea-aie-project/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.258400,0.208691,0.920000,0.891577
2,0.148300,0.157557,0.934500,0.908507
3,0.115200,0.153803,0.938500,0.914395


/Users/v.razon/Desktop/вуз/ml/mirea-aie-project/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/v.razon/Desktop/вуз/ml/mirea-aie-project/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=1500, training_loss=0.30235091876983644, metrics={'train_runtime': 1579.8811, 'train_samples_per_second': 30.382, 'train_steps_per_second': 0.949, 'total_flos': 1589722177536000.0, 'train_loss': 0.30235091876983644, 'epoch': 3.0})

In [14]:
# Результаты на validation (лучшая эпоха)
val_results = trainer.evaluate()
print("Validation results:")
for k, v in val_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

/Users/v.razon/Desktop/вуз/ml/mirea-aie-project/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Validation results:
  eval_loss: 0.1538
  eval_accuracy: 0.9385
  eval_f1_macro: 0.9144
  eval_runtime: 20.6544
  eval_samples_per_second: 96.8320
  eval_steps_per_second: 1.5490
  epoch: 3.0000


## 6. Оценка на test и анализ ошибок

In [15]:
# Финальная оценка на test (один раз)
test_results = trainer.evaluate(tokenized_dataset["test"])
print("Test results:")
for k, v in test_results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

/Users/v.razon/Desktop/вуз/ml/mirea-aie-project/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test results:
  eval_loss: 0.1710
  eval_accuracy: 0.9260
  eval_f1_macro: 0.8815
  eval_runtime: 20.7691
  eval_samples_per_second: 96.2970
  eval_steps_per_second: 1.5410
  epoch: 3.0000


In [16]:
# Предсказания на test
test_predictions = trainer.predict(tokenized_dataset["test"])
preds = np.argmax(test_predictions.predictions, axis=-1)
true_labels = test_predictions.label_ids

test_accuracy = accuracy_score(true_labels, preds)
test_f1 = f1_score(true_labels, preds, average="macro")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test F1 macro: {test_f1:.4f}")
print()
print(classification_report(true_labels, preds, target_names=label_names))

/Users/v.razon/Desktop/вуз/ml/mirea-aie-project/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Test accuracy: 0.9260
Test F1 macro: 0.8815

              precision    recall  f1-score   support

     sadness       0.97      0.96      0.96       581
         joy       0.95      0.94      0.94       695
        love       0.79      0.84      0.82       159
       anger       0.92      0.94      0.93       275
        fear       0.89      0.92      0.90       224
    surprise       0.81      0.67      0.73        66

    accuracy                           0.93      2000
   macro avg       0.89      0.88      0.88      2000
weighted avg       0.93      0.93      0.93      2000



In [17]:
# Матрица ошибок
cm = confusion_matrix(true_labels, preds)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix (Test)")
plt.tight_layout()
plt.savefig("artifacts/confusion_matrix.png", dpi=150)
plt.show()
print("Saved: artifacts/confusion_matrix.png")

Saved: artifacts/confusion_matrix.png


/var/folders/n_/mqpqp2w54_n0gnng_thc3sv40000gq/T/ipykernel_80093/313012021.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# Примеры предсказаний — фильтруем пустые тексты
test_texts = dataset["test"]["text"]
confidences = torch.softmax(
    torch.tensor(test_predictions.predictions), dim=-1
).max(dim=-1).values.numpy()

df_preds = pd.DataFrame({
    "text": test_texts,
    "true_label": [label_names[l] for l in true_labels],
    "pred_label": [label_names[p] for p in preds],
    "confidence": np.round(confidences, 4),
})

# Убираем строки с пустым или слишком коротким текстом
df_preds = df_preds[df_preds["text"].str.strip().str.len() > 5].reset_index(drop=True)

# 10 правильных + 10 ошибок
correct = df_preds[df_preds["true_label"] == df_preds["pred_label"]].head(10)
errors = df_preds[df_preds["true_label"] != df_preds["pred_label"]].head(10)
sample_df = pd.concat([correct, errors]).reset_index(drop=True)
sample_df.to_csv("artifacts/sample_predictions.csv", index=False)
print("Saved: artifacts/sample_predictions.csv")
print(f"Total errors: {(df_preds['true_label'] != df_preds['pred_label']).sum()} / {len(df_preds)}")

Saved: artifacts/sample_predictions.csv
Total errors: 148 / 2000


In [19]:
# Примеры ошибок модели
print("Примеры ошибок модели:")
print("=" * 80)
error_df = df_preds[df_preds["true_label"] != df_preds["pred_label"]].head(10)
for _, row in error_df.iterrows():
    print(f"Text: {row['text'][:120]}")
    print(f"  True: {row['true_label']} | Pred: {row['pred_label']} | Conf: {row['confidence']:.3f}")
    print()

Примеры ошибок модели:
Text: i don t feel particularly agitated
  True: fear | Pred: anger | Conf: 0.870

Text: i am right handed however i play billiards left handed naturally so me trying to play right handed feels weird
  True: surprise | Pred: fear | Conf: 0.650

Text: i feel like i am in paradise kissing those sweet lips make me feel like i dive into a magical world of love
  True: joy | Pred: love | Conf: 0.510

Text: when a friend dropped a frog down my neck
  True: anger | Pred: fear | Conf: 0.669

Text: i feel agitated with myself that i did not foresee her frustrations earlier leading to the ending of our relationship
  True: fear | Pred: anger | Conf: 0.529

Text: i looked at mabel this morning i named my left breast mabel my right one is hazel and i feel this weird mixture of anger
  True: fear | Pred: surprise | Conf: 0.542

Text: i feel very mislead by someone that i really really thought i knew and liked very much so
  True: love | Pred: anger | Conf: 0.831

Text: im fee

## Краткий анализ ошибок

Типичные ошибки связаны с семантически близкими эмоциями:
- **sadness/anger**: оба класса выражают негативные эмоции, граница размыта
- **love/joy**: оба позитивные, часто встречаются в схожих контекстах
- **fear/sadness**: тревога и грусть пересекаются в описаниях трудных ситуаций
- **surprise**: малочисленный класс (3.6% train) — модель хуже его распознаёт

Модель уверенно различает joy и sadness (наиболее частые и лексически чёткие классы), но путается на редких классах и пограничных случаях.